# Working with parquet files

## Objective

+ In this assignment, we will use the data downloaded with the module `data_manager` to create features.

(11 pts total)

## Prerequisites

+ This notebook assumes that price data is available to you in the environment variable `PRICE_DATA`. If you have not done so, then execute the notebook `01_materials/labs/2_data_engineering.ipynb` to create this data set.


+ Load the environment variables using dotenv. (1 pt)

In [8]:
# Write your code below.
%load_ext dotenv
%dotenv 

The dotenv extension is already loaded. To reload it, use:
  %reload_ext dotenv


In [9]:
import dask.dataframe as dd

+ Load the environment variable `PRICE_DATA`.
+ Use [glob](https://docs.python.org/3/library/glob.html) to find the path of all parquet files in the directory `PRICE_DATA`.

(1pt)

In [20]:
import os
from glob import glob

PRICE_DATA = os.getenv("PRICE_DATA")
# Write your code below.
parquet_files = glob(os.path.join(PRICE_DATA, "**/*.parquet"), recursive = True)
dd_px = dd.read_parquet(parquet_files).set_index("Ticker")

For each ticker and using Dask, do the following:

+ Add lags for variables Close and Adj_Close.
+ Add returns based on Close:
    
    - `returns`: (Close / Close_lag_1) - 1

+ Add the following range: 

    - `hi_lo_range`: this is the day's High minus Low.

+ Assign the result to `dd_feat`.

(4 pt)

In [22]:
# Checking to make sure index is set as the column "Ticker"
dd_px.index.name

'Ticker'

In [29]:
dd_px.head()
# Adj_Close is Adj Close

,Date,Adj Close,Close,High,Low,Open,Volume,Year
Ticker,,,,,,,,
AAPL,2000-01-03 00:00:00+00:00,0.843077,0.999442,1.004464,0.907924,0.936384,535796800.0,2000
AAPL,2018-05-11 00:00:00+00:00,44.718655,47.147499,47.514999,46.862499,47.372501,104848800.0,2018
AAPL,2018-05-10 00:00:00+00:00,44.889385,47.509998,47.592499,46.912498,46.935001,111957200.0,2018
AAPL,2018-05-09 00:00:00+00:00,44.256344,46.840000,46.849998,46.305000,46.637501,92844800.0,2018
AAPL,2018-07-03 00:00:00+00:00,43.611301,45.980000,46.987499,45.884998,46.947498,55819200.0,2018


In [34]:
# Write your code below.
dd_feat = dd_px.groupby('Ticker', group_keys=False).apply(
    lambda x: x.assign(
        Close_lag_1 = x['Close'].shift(1),
        Adj_Close_lag_1 = x['Adj Close'].shift(1)
))

dd_feat = dd_feat.assign(
    returns = (dd_feat['Close'] - dd_feat['Close_lag_1']) - 1
)

dd_feat = dd_feat.assign(
    hi_lo_range = dd_feat['High'] - dd_feat['Low']
)

C:\Users\Ken\AppData\Local\Temp\ipykernel_7976\1468233509.py:2: UserWarning: `meta` is not specified, inferred from partial data. Please provide `meta` if the result is unexpected.
  Before: .apply(func)
  After:  .apply(func, meta={'x': 'f8', 'y': 'f8'}) for dataframe result
  or:     .apply(func, meta=('x', 'f8'))            for series result
  dd_feat = dd_px.groupby('Ticker', group_keys=False).apply(


In [35]:
# Check
dd_feat.head()

,Date,Adj Close,Close,High,Low,Open,Volume,Year,Close_lag_1,Adj_Close_lag_1,returns,hi_lo_range
Ticker,,,,,,,,,,,,
AAPL,2000-01-03 00:00:00+00:00,0.843077,0.999442,1.004464,0.907924,0.936384,535796800.0,2000,NaN,NaN,NaN,0.096540
AAPL,2000-01-04 00:00:00+00:00,0.771997,0.915179,0.987723,0.903460,0.966518,512377600.0,2000,0.999442,0.843077,-1.084263,0.084263
AAPL,2000-01-05 00:00:00+00:00,0.783293,0.928571,0.987165,0.919643,0.926339,778321600.0,2000,0.915179,0.771997,-0.986608,0.067522
AAPL,2000-01-06 00:00:00+00:00,0.715508,0.848214,0.955357,0.848214,0.947545,767972800.0,2000,0.928571,0.783293,-1.080357,0.107143
AAPL,2000-01-07 00:00:00+00:00,0.749402,0.888393,0.901786,0.852679,0.861607,460734400.0,2000,0.848214,0.715508,-0.959821,0.049107


+ Convert the Dask data frame to a pandas data frame. 
+ Add a new feature containing the moving average of `returns` using a window of 10 days. There are several ways to solve this task, a simple one uses `.rolling(10).mean()`.

(3 pt)

In [47]:
# Write your code below.
df_px = dd_feat.compute()

# Check
type(df_px)

pandas.core.frame.DataFrame

In [49]:
df_px['moving_average'] = df_px.groupby('Ticker')['returns'].transform(lambda x: x.rolling(10).mean())

In [51]:
df_px

,Date,Adj Close,Close,High,Low,Open,Volume,Year,Close_lag_1,Adj_Close_lag_1,returns,hi_lo_range,moving_average
Ticker,,,,,,,,,,,,,
AAPL,2000-01-03 00:00:00+00:00,0.843077,0.999442,1.004464,0.907924,0.936384,535796800.0,2000,NaN,NaN,NaN,0.096540,NaN
AAPL,2000-01-04 00:00:00+00:00,0.771997,0.915179,0.987723,0.903460,0.966518,512377600.0,2000,0.999442,0.843077,-1.084263,0.084263,NaN
AAPL,2000-01-05 00:00:00+00:00,0.783293,0.928571,0.987165,0.919643,0.926339,778321600.0,2000,0.915179,0.771997,-0.986608,0.067522,NaN
AAPL,2000-01-06 00:00:00+00:00,0.715508,0.848214,0.955357,0.848214,0.947545,767972800.0,2000,0.928571,0.783293,-1.080357,0.107143,NaN
AAPL,2000-01-07 00:00:00+00:00,0.749402,0.888393,0.901786,0.852679,0.861607,460734400.0,2000,0.848214,0.715508,-0.959821,0.049107,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...
ZBRA,2025-01-17 00:00:00+00:00,405.709991,405.709991,407.290009,402.290009,406.040009,270600.0,2025,402.720001,402.720001,1.989990,5.000000,1.194998
ZBRA,2025-01-21 00:00:00+00:00,418.070007,418.070007,419.850006,407.619995,407.619995,446000.0,2025,405.709991,405.709991,11.360016,12.230011,1.621002
ZBRA,2025-01-22 00:00:00+00:00,420.570007,420.570007,427.760010,419.589996,425.239990,497500.0,2025,418.070007,418.070007,1.500000,8.170013,1.524002


Please comment:

+ Was it necessary to convert to pandas to calculate the moving average return?

    **Answer:** No, it was not necessary to convert to pandas to calculate the moving average return. Dask can support rolling() functions as well, as long as the DataFrame has known index divisons, which this DataFrame does. 

+ Would it have been better to do it in Dask? Why?

    **Answer:** It would have been better if the dataset was larger, but this dataset had around 400,000 rows, which pandas would be able to handle. If the dataset was larger, perhaps over 1 million rows, then Dask would have been better as it can scale to bigger datasets. 

So in this case, it would not have been better to do in Dask.

(1 pt)

In [56]:
# Check divisions in Dask dataframe
# dd_feat.divisions


## Criteria

The [rubric](./assignment_1_rubric_clean.xlsx) contains the criteria for grading.

## Submission Information

🚨 **Please review our [Assignment Submission Guide](https://github.com/UofT-DSI/onboarding/blob/main/onboarding_documents/submissions.md)** 🚨 for detailed instructions on how to format, branch, and submit your work. Following these guidelines is crucial for your submissions to be evaluated correctly.

### Submission Parameters:
* Submission Due Date: `HH:MM AM/PM - DD/MM/YYYY`
* The branch name for your repo should be: `assignment-1`
* What to submit for this assignment:
    * This Jupyter Notebook (assignment_1.ipynb) should be populated and should be the only change in your pull request.
* What the pull request link should look like for this assignment: `https://github.com/<your_github_username>/production/pull/<pr_id>`
    * Open a private window in your browser. Copy and paste the link to your pull request into the address bar. Make sure you can see your pull request properly. This helps the technical facilitator and learning support staff review your submission easily.

Checklist:
- [x] Created a branch with the correct naming convention.
- [x] Ensured that the repository is public.
- [x] Reviewed the PR description guidelines and adhered to them.
- [x] Verify that the link is accessible in a private browser window.

If you encounter any difficulties or have questions, please don't hesitate to reach out to our team via our Slack at `#cohort-3-help`. Our Technical Facilitators and Learning Support staff are here to help you navigate any challenges.